## 1. 라이브러리 설치

In [ ]:
# Colab 첫 셀
!pip install openai pypdf tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 기본 설정 & Upstage 클라이언트

In [ ]:
import os
from typing import List, Tuple
import numpy as np
from tqdm import tqdm
from pypdf import PdfReader
from openai import OpenAI

# Upstage API Key 직접 입력
UPSTAGE_API_KEY = "your-api-key"

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# 사용할 임베딩 모델 (MMLU 위키 만들 때 쓴 모델과 맞추는 게 제일 좋음)
EMBED_MODEL = "embedding-passage"

# 파일 경로 (ewha.pdf 업로드)
EWHA_PDF_PATH = "/content/drive/MyDrive/nlp_me/data/ewha.pdf"
EWHA_EMB_NPZ = "/content/drive/MyDrive/nlp_me/result/result_ewha/ewha_embeddings.npz"


## 3. PDF에서 텍스트 추출

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> List[str]:
    """
    PDF에서 각 페이지의 텍스트를 리스트로 반환.
    page_texts[i] = i번째 페이지의 전체 텍스트
    """
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ""
        # 공백 정리
        text = text.replace("\r", "\n")
        pages.append(text)
    return pages

page_texts = extract_text_from_pdf(EWHA_PDF_PATH)
print(f"총 페이지 수: {len(page_texts)}")
print(page_texts[0][:500])  # 첫 페이지 앞부분 확인


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/nlp_me/data/ewha.pdf'

## 텍스트 청크 나누기

In [ ]:
def split_into_chunks(
    pages: List[str],
    max_chars: int = 700,
    min_chars: int = 200
) -> List[str]:
    """
    페이지 리스트를 받아, 줄/문단 기준으로 나눈 뒤
    max_chars 근처가 되도록 청크를 묶어서 반환.
    """
    chunks: List[str] = []
    cur = ""

    for p_idx, page in enumerate(pages):
        # 빈 페이지는 스킵
        if not page.strip():
            continue

        # 줄 단위로 분리
        lines = [ln.strip() for ln in page.split("\n")]
        lines = [ln for ln in lines if ln]  # 빈 줄 제거

        for ln in lines:
            # 현재 청크에 줄 추가했을 때 너무 길면, 기존 청크를 저장하고 새로 시작
            if len(cur) + len(ln) + 1 > max_chars:
                if len(cur) >= min_chars:
                    chunks.append(cur.strip())
                    cur = ln
                else:
                    # cur가 너무 짧은데 max 넘으면 그냥 붙여버리기
                    cur += " " + ln
            else:
                if cur:
                    cur += " " + ln
                else:
                    cur = ln

        # 페이지 끝나면, cur가 적당히 길면 청크로 저장
        if len(cur) >= min_chars:
            chunks.append(cur.strip())
            cur = ""

    # 마지막 남은 청크도 추가
    if cur.strip():
        chunks.append(cur.strip())

    return chunks

chunks = split_into_chunks(page_texts, max_chars=700, min_chars=200)
print(f"생성된 청크 개수: {len(chunks)}")
print("첫 번째 청크 예시:\n", chunks[0])


생성된 청크 개수: 56
첫 번째 청크 예시:
 2 - 2 - 1 이화여자대학교 학칙1946. 8. 15.  제정2017. 8. 16.  개정제1장 총칙제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과 그 광범하고 정밀한 응용방법을 교수․연구하며, 인격을 도야하여 국가와 인류사회의 발전에 공헌할 수 있는 지도여성을 양성함을 목적으로 한다.제2조(명칭) 본교는 이화여자대학교라 부른다.제3조(위치) 본교는 서울특별시 서대문구 이화여대길 52에 둔다. (개정 2013.2.25.)제2장 편제제4조(대학 및 대학원) ① 본교에는 다음 각 호의 대학을 둔다.  1. 인문과학대학, 사회과학대학, 자연과학대학, 엘텍공과대학, 음악대학, 조형예술대학, 사범대학, 경영대학, 신산업융합대학, 의과대학, 간호대학, 약학대학, 스크랜튼대학(이하 “각 대학”이라 한다)  (개정 2016.6.16.)  2. 호크마(HOKMA)교양대학② 본교에는 대학원, 국제대학원, 통역번역대학원, 경영전문대학원, 법학전문대학원, 교육대학원, 디자인대학원,  사회복지대학원, 신학대학원, 정책과학대학원, 공연예술대학원, 임상보건융합대학원, 임상치의학대학원, 외국어교육특수대학원을 둔다(이하 “각 대학원”이라 한다).  (개정 2016.6.16., 2017.5.15.)[전문개정  2015.11.27.]제5조(학부․학과․전공 및 정원) ① 각 대학, 학부, 학과, 전공 및 모집단위별 입학정원은 별표 1과 같다.  (개정 2015.5.8., 2016.2.16., 2016.2.26., 2016.5.19., 2017.5.4., 2017.5.15.)② 모집단위별 입학정원의 일부는 입학전형에 따라 2개 이상의 모집단위를 통합하여 모집할 수 있다. (개정 1999.2.9., 2017.5.15.)③ 제2항에 따라 통합된 모집단위로 입학한 학생과 대학 또는 학부 등 광역화된 모집단위로 입학한 학생에 대하여는 일정한 학기와 학점을 이수한 후에 총장의 승인을 얻어 이수할 전공을 결정하게 하

## 5. Upstage 임베딩 생성 (배치 처리)

In [ ]:
def embed_texts_batch(
    texts: List[str],
    model: str = EMBED_MODEL,
    batch_size: int = 16
) -> np.ndarray:
    """
    여러 텍스트를 배치로 나눠 Upstage 임베딩 API 호출.
    결과 shape = (len(texts), d)
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(
            model=model,
            input=batch
        )
        # resp.data[j].embedding 리스트를 배열로 변환
        for item in resp.data:
            all_embeddings.append(item.embedding)

    emb_array = np.array(all_embeddings, dtype="float32")
    return emb_array

ewha_embeddings = embed_texts_batch(chunks, model=EMBED_MODEL, batch_size=16)
print("임베딩 shape:", ewha_embeddings.shape)


100%|██████████| 4/4 [00:09<00:00,  2.35s/it]

임베딩 shape: (56, 4096)


## 6. npz로 저장 (embeddings + chunks)

In [ ]:
def save_kb_npz(embeddings: np.ndarray, chunks: List[str], out_path: str):
    """
    embeddings, chunks를 npz 형식으로 저장
    - embeddings: (N, d) float32
    - chunks: (N,) object / str
    """
    # npz는 배열만 저장 가능하니까, chunks는 object 배열로 변환
    chunk_arr = np.array(chunks, dtype=object)
    np.savez(out_path, embeddings=embeddings, chunks=chunk_arr)
    print(f"Saved KB to {out_path}")
    print(" - embeddings:", embeddings.shape)
    print(" - chunks:", chunk_arr.shape)

save_kb_npz(ewha_embeddings, chunks, EWHA_EMB_NPZ)


✅ Saved KB to /content/drive/MyDrive/nlp_me/result/result_ewha/ewha_embeddings.npz
 - embeddings: (56, 4096)
 - chunks: (56,)


## 7. 저장 여부 테스트 로드

In [ ]:
# 저장된 npz를 다시 불러서 확인
data = np.load(EWHA_EMB_NPZ, allow_pickle=True)
print("로드한 embeddings shape:", data["embeddings"].shape)
print("로드한 chunks 개수:", len(data["chunks"]))
print("예시 chunk:\n", data["chunks"][0][:200])


🔍 로드한 embeddings shape: (56, 4096)
🔍 로드한 chunks 개수: 56
예시 chunk:
 2 - 2 - 1 이화여자대학교 학칙1946. 8. 15.  제정2017. 8. 16.  개정제1장 총칙제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과 그 광범하고 정밀한 응용방법을 교수․연구하며, 인격을 도야하여 국가와 인류사회의 발전에 공헌할 수 있는 지도여성을 양성함을 목적으로 한다.제2조(명칭) 본교는 이화여
